<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/tf_idf_key_authors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
try:
    # Running inside Google Colab → use !pip
    import google.colab
    from google.colab import drive
    in_google_colab = True
    print("Running in Colab → installing packages...")
    !pip install scikit-learn
except ImportError:
    # Not running in Colab
    in_google_colab = False
    pass

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

from pathlib import Path

import networkx as nx
import matplotlib.pyplot as plt

# Loading and Exporting

## Parameters

In [ ]:
dataset_name = "Labeled_Datasets/Ecuador"   # folder path
file_name = "Ecuador_part_5.gzip.parquet"   # file name
dataset_procced_date = "2025_12_01"         # ("%Y_%m_%d")

# Recommended: working directory is: "Backbone-Graph/src"

## Load Data

In [ ]:
from pathlib import Path

# Google Colab
if in_google_colab:
    raise Exception("Google Colab not supported yet.")
    print("Running inside Colab. Loading from Google Drive...")

# Cluster\Local
else:
    # Recommended: working directory is: "Backbone-Graph/src"
    # Loading result of "Topic_Clustring_and_NER.ipynb".
    file_name_no_suffixes = file_name.split(".")[0]
    processed_dataset_path = Path("../results") / dataset_name / file_name_no_suffixes / dataset_procced_date
    # Example: ""../results/Labeled_Datasets/Ecuador/Ecuador_part_4/2025_11_27"
    print(f"Running on local/cluster.\nLoading data from: {processed_dataset_path}")


# Load the parquet file
df = pd.read_parquet(processed_dataset_path)

## Output

In [ ]:
output_path = processed_dataset_path # save in the save folder as the processed dataset

# Ley Entities

TF-IDF Using *sklearn*

In [ ]:

def compute_sklearn_tfidf(df, entity_col, topic_col='topic'):
    """
    Compute TF-IDF over entities grouped by topic.
    Args:
        df (pd.DataFrame): Must contain [topic_col] and [entity_col] (list of entities).
        entity_col (str): Column containing lists of entities per post.
        topic_col (str): Topic label column.
    Returns:
        pd.DataFrame: TF-IDF matrix (rows = topics, cols = entities).
    """

    # 1) Explode entity lists → one row per (topic, entity)
    df_exp = (
        df[[topic_col, entity_col]]
        .explode(entity_col)
        .dropna(subset=[entity_col])
        .astype({entity_col: str})
    )

    # 2) Build "documents" for each topic = all entities of that topic joined into a single string
    topic_docs_df = (
        df_exp.groupby(topic_col)[entity_col]
              .apply(lambda x: " ".join(x))
              .reset_index()
              .rename(columns={entity_col: "doc"})
    )

    # 3) Run TF-IDF
    vec = TfidfVectorizer(token_pattern=r'\S+')  # treat each entity as token
    X = vec.fit_transform(topic_docs_df["doc"])

    tfidf_df = pd.DataFrame(
        X.toarray(),
        columns=vec.get_feature_names_out()
    )

    # 4) Insert topic as first column
    tfidf_df.insert(0, topic_col, topic_docs_df[topic_col].values)

    return tfidf_df

In [ ]:
tfidf_df = compute_sklearn_tfidf(df, entity_col='entities')
tfidf_df.head()

In [ ]:
def get_top_entities(tfidf_df, topic_id, topic_col, n=10):
    """
    Return the top n entities with the highest TF-IDF scores for a given topic.
    Args:
        tfidf_df (pd.DataFrame): df ['rows=topics, collums = enteties, cells_values = tf-idf score]
                                 Must include a 'topic' column.
        topic_lable: The topic name or id.
        n (int): Number of top entities to return (default=10).
        topic_col (str): Name of the topic column (default='topic').
    Returns:
        top_entities (pd.Series): Index = entity names, Values = TF-IDF scores (sorted descending).
    """
    # find the row corresponding to the topic
    topic_enteties_df = tfidf_df.loc[tfidf_df[topic_col] == topic_id]

    # remove the topic column and get entity scores as Series (index = entetity, values = tf-idf score]
    entity_scores_series = topic_enteties_df.drop(columns=[topic_col]).iloc[0]

    # sort entities by score and return top-n
    top_entities = entity_scores_series.sort_values(ascending=False).head(n)

    return top_entities

In [ ]:
top_entities = get_top_entities(tfidf_df, topic_id=1, topic_col = "BERTopic_topic" n=10)
top_entities

# Networkx Graphs

## Build Bipartite Account ↔ Account Graph

In [ ]:
def add_relation_edges(
    G: nx.Graph,
    df: pd.DataFrame,
    src_col: str,
    dst_col: str,
    weight: int = 1,
):
    """Add edges to a graph from df[src_col] → df[dst_col].

    Automatically explodes src or dst if they contain list-like values.

    Args:
        G (nx.Graph): Graph to update (modified in-place).
        df (pd.DataFrame): Input DataFrame.
        src_col (str): Column containing sources (single or list-like).
        dst_col (str): Column containing targets (single or list-like).
        weight (int): Weight added per relation (default = 1).

    Returns:
        None: Graph is modified directly.
    """

    # keep valid rows
    pairs = df[[src_col, dst_col]].dropna()

    # auto-detect list-like columns
    is_src_list = pairs[src_col].map(lambda x: isinstance(x, (list, tuple, set))).any()
    is_dst_list = pairs[dst_col].map(lambda x: isinstance(x, (list, tuple, set))).any()

    # explode step-by-step
    if is_src_list:
        pairs = pairs.explode(src_col).dropna(subset=[src_col])
    if is_dst_list:
        pairs = pairs.explode(dst_col).dropna(subset=[dst_col])

    # aggregate (src, dst) counts
    counts = (
        pairs
        .value_counts([src_col, dst_col])
        .reset_index(name="count")
    )

    # build weighted edges
    for _, row in counts.iterrows():
        src = row[src_col]
        dst = row[dst_col]
        w = row["count"] * weight

        if src not in G:
            G.add_node(src)
        if dst not in G:
            G.add_node(dst)

        if G.has_edge(src, dst):
            G[src][dst]["weight"] += w
        else:
            G.add_edge(src, dst, weight=w)

In [ ]:
account_relations = [
    ("accountid", "in_reply_to_accountid"),   # reply
    ("accountid", "account_mentions"),        # mentions (list)
    ("accountid", "reposted_accountid"),      # repost
]

In [ ]:
G_accounts = nx.DiGraph()

for src_col, dst_col in account_relations:

    add_relation_edges(
        G_accounts,
        df,
        src_col=src_col,
        dst_col=dst_col,
        weight=1,
    )

## Plot Graph

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path

def plot_and_save_graph(
    G: nx.Graph,
    output_path: str = "",
    graph_name: str = "graph",
    figsize=(12, 12),
):
    """Plot a NetworkX graph, display it, and save it if provided with output_path as a PNG file.

    Args:
        G (nx.Graph): Graph to visualize.
        output_path (str): Directory where the image will be saved (default: "").
        graph_name (str): Base name of the saved file (default: 'graph').
        figsize (tuple): Figure size in inches.

    Returns:
        None
    """
    plt.figure(figsize=figsize)

    # node positions (stable layout)
    pos = nx.spring_layout(G, seed=42)

    # draw graph
    nx.draw(
        G,
        pos,
        node_size=30,
        node_color="steelblue",
        edge_color="gray",
        width=0.5,
        alpha=0.8,
        with_labels=False,
    )

    plt.tight_layout()

    # build output path
    if output_path:
      plot_path = Path(save_dir) / f"{graph_name}.png"
      plt.savefig(plot_path, dpi=300)
      print(f"Saved graph to: {plot_path}")